In [5]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.data_processing import (
    extract_condition_coc, 
    extract_day_coc,
    extract_hours_prior_coc
)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

COND_ORDER = ["+FGF", "-FGF"]
CONDITION_PALETTE = {"+FGF": "#2E86AB", "-FGF": "#A23B72"}
MIN_MAX_CELL_VOLUME = (500, 6500) #seems safe after exploring the volume distribution

In [3]:
# Notebook runs from notebooks/, so results is one level up
results_dir = Path("../coc_results").resolve()
output_dir = results_dir / "processed_csvs"
output_file = output_dir / "processed_coc.csv"

# Find every CSV under results/ recursively, excluding the final output file if it already exists
csv_files = sorted(
    csv_path
    for csv_path in results_dir.rglob("*.csv")
    if csv_path.resolve() != output_file.resolve()
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {results_dir}")

# Read and merge all CSV files
merged_df = pd.concat((pd.read_csv(csv_file) for csv_file in csv_files), ignore_index=True)

# Extract 'condition', 'day' and 'magnification' metadata columns and store them
merged_df["condition"] = merged_df["lif_image_name"].apply(extract_condition_coc)
merged_df["day"] = merged_df["lif_container_id"].apply(extract_day_coc)
merged_df["hours_prior_toHPFa"] = merged_df["lif_container_id"].apply(extract_hours_prior_coc)

# Ensure output directory exists and save merged CSV (unfiltered)
output_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_file, index=False)

print(f"Merged {len(csv_files)} CSV files into: {output_file}")
print(f"Merged dataframe shape: {merged_df.shape}")
merged_df.head()

Merged 23 CSV files into: C:\Users\adiez_cmic\github_repos\margrecs_alginate\coc_results\processed_csvs\processed_coc.csv
Merged dataframe shape: (40633, 31)


,lif_container_id,lif_image_name,population,label,centroid-0,centroid-1,centroid-2,area,area_bbox,area_filled,...,HPFa_red_mean_int,HPFa_red_min_int,HPFa_red_max_int,HPFa_red_std_int,HPFa_red_sum_int,%_area_cleaned,input_img_shape,condition,day,hours_prior_toHPFa
0,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,A549_green,1,3.266212,6.088737,53.336177,586.0,1287.0,586.0,...,61.890785,13.0,210.0,33.384742,36268.0,NaN,"[137, 512, 512]",+FGF,1,24
1,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,A549_green,2,0.888889,2.472222,72.361111,36.0,60.0,36.0,...,35.694444,16.0,69.0,12.135392,1285.0,NaN,"[137, 512, 512]",+FGF,1,24
2,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,A549_green,3,4.934562,5.679263,238.174194,1085.0,3952.0,1085.0,...,77.576959,14.0,255.0,47.277851,84171.0,NaN,"[137, 512, 512]",+FGF,1,24
3,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,A549_green,4,0.972727,12.645455,72.309091,110.0,256.0,110.0,...,44.090909,15.0,120.0,24.926271,4850.0,NaN,"[137, 512, 512]",+FGF,1,24
4,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,A549_green,5,3.477612,11.248756,240.149254,201.0,432.0,201.0,...,37.353234,14.0,82.0,13.314773,7508.0,NaN,"[137, 512, 512]",+FGF,1,24


In [9]:
# Filter out rows where area is below 500 or above 6500
filtered_df = merged_df[(merged_df["area"] >= MIN_MAX_CELL_VOLUME[0]) & (merged_df["area"] <= MIN_MAX_CELL_VOLUME[1])]

# Total cell number after filtering (per image)
total_cells_df = (
    filtered_df.groupby(["lif_container_id", "lif_image_name", "population", "day", "condition", "hours_prior_toHPFa"], as_index=False)
    .size()
    .rename(columns={"size": "total_cell_nr"})
)

total_cells_file = output_dir / "total_cell_nr.csv"
total_cells_df.to_csv(total_cells_file, index=False)

# Assign filtered_df to merged_df to ensure all plots use the same source of truth
merged_df = filtered_df

In [10]:
total_cells_df

,lif_container_id,lif_image_name,population,day,condition,hours_prior_toHPFa,total_cell_nr
0,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,A549_green,1,+FGF,24,891
1,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x,HPFa_red,1,+FGF,24,69
2,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x2,A549_green,1,+FGF,24,884
3,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-1.1-10x2,HPFa_red,1,+FGF,24,123
4,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-2.3-10x,A549_green,1,+FGF,24,622
5,Co-culture-day1-A549-24hpriortoHPFa,D1SD1+FGF-2.3-10x,HPFa_red,1,+FGF,24,428
6,Co-culture-day1-A549-24hpriortoHPFa,D1SD1-FGF-1.1-10x,A549_green,1,-FGF,24,1073
7,Co-culture-day1-A549-24hpriortoHPFa,D1SD1-FGF-1.1-10x,HPFa_red,1,-FGF,24,88
8,Co-culture-day1-A549-24hpriortoHPFa,D1SD1-FGF-1.1-10x2,A549_green,1,-FGF,24,889
9,Co-culture-day1-A549-24hpriortoHPFa,D1SD1-FGF-1.1-10x2,HPFa_red,1,-FGF,24,121
